In [ ]:
import os, base64
from dotenv import load_dotenv
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed

# Load API keys from .env file
load_dotenv()
API_KEYS = [os.getenv(f"API_KEY_{i}") for i in range(1, 41)]

# Create 40 OpenAI clients with different API keys
clients = [OpenAI(api_key=key, base_url="https://api.moonshot.cn/v1") for key in API_KEYS]

# Input and output paths
data_collection_path = r"C:\Users\lby13\Desktop\data collection&labelling\data_collection"
data_label_path = r"C:\Users\lby13\Desktop\data collection&labelling\data_label"
os.makedirs(data_label_path, exist_ok=True)

In [ ]:
def recognize_captcha(img_num, client):
    """
    Call the API to recognize a captcha image and return the result.
    """
    image_path = os.path.join(data_collection_path, f"img{img_num}.png")
    
    with open(image_path, "rb") as f:
        image_base64 = base64.b64encode(f.read()).decode('utf-8')
    
    response = client.chat.completions.create(
        model="kimi-k2.6",
        messages=[
            {"role": "system", "content": "You are a captcha recognition assistant. Return only the captcha content."},
            {"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_base64}"}},
                {"type": "text", "text": "Recognize this 4-character captcha. Return exactly 4 characters (letters/digits only)."},
            ]},
        ],
    )
    
    return response.choices[0].message.content.strip()


def process_images(start, end):
    """
    Process captcha images in parallel using 40 threads with different API clients.
    """
    tasks = list(range(start, end))
    total = len(tasks)
    success = 0
    fail = 0
    
    # Create thread pool with 40 workers
    with ThreadPoolExecutor(max_workers=40) as executor:
        # Submit tasks, each assigned to a client by round-robin
        futures = {
            executor.submit(recognize_captcha, img_num, clients[img_num % 40]): img_num
            for img_num in tasks
        }
        
        # Collect results as they complete
        for future in as_completed(futures):
            img_num = futures[future]
            captcha_text = future.result()
            # Save result to text file
            txt_path = os.path.join(data_label_path, f"img{img_num}.txt")
            with open(txt_path, 'w', encoding='utf-8') as f:
                f.write(captcha_text)

In [ ]:
# Process images: img1.png ~ img5000.png
process_images(1, 5001)